# NB18 — Geochemical Niche Breadth

Tests whether metal gene-dense genera occupy **narrower ranges of environmental metal concentrations** — a more direct measure of geochemical specialization than categorical Levins' B.

**Geochemical niche width**: SD of metal concentration (Cu, Zn, Pb, Ni, Co, As, Cr, pH) across genus detection sites. Low SD = geochemical specialist; high SD = geochemical generalist.

**Three analyses:**
1. MGnify MAGs (n=22,356 globally annotated genomes; NGSA + CMMI metal databases)
2. AusMicrobiome 16S (BASE; NGSA annotation; Cu/Zn/Pb/Ni/Co/As/Cr/Hg/pH)
3. Hotspot occupancy — fraction of MAG detections in metal-enriched hotspot cells

PGLS run for each metal independently; FDR correction (BH) across metals within each dataset.

In [1]:
import os, subprocess, tempfile, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

BASE    = '/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_metal_ecology'
TREE    = f'{BASE}/data/gtdb_bac_genus_pruned.tree'
PGLS_R  = f'{BASE}/scripts/pgls_mgnify_validation.R'
FIGDIR  = f'{BASE}/data/figures'
RSCRIPT = '/home/hmacgregor/.local/envs/bio_env/bin/Rscript'
os.makedirs(FIGDIR, exist_ok=True)
os.chdir(BASE)

# ── colour palette (consistent with NB17) ──────────────────────────────────
METAL_COLORS = {
    'Cu': '#c0392b', 'Zn': '#e67e22', 'Pb': '#8e44ad',
    'Ni': '#2980b9', 'Co': '#27ae60', 'As': '#d35400',
    'Cr': '#16a085', 'Hg': '#7f8c8d', 'pH': '#2c3e50',
}
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete.')

Setup complete.


## Block 1 — MGnify MAG geochemical niche width (all metals)

In [2]:
# Row-aligned join: final_mags_geospatial_traits (genus) + mags_annotated_geo (NGSA/CMMI metals)
mag_traits = pd.read_csv('data/final_mags_geospatial_traits.csv',
                         usecols=['genome_id', 'genus', 'lat', 'lon',
                                  'n_metal_types', 'total_metal_genes'])
mag_geo    = pd.read_csv('data/mags_annotated_geo.csv')

# Sanity check: row-aligned?
assert (mag_traits['lat'] == mag_geo['lat']).all(), 'Row alignment broken'

mags = pd.concat([mag_traits, mag_geo.drop(columns=['lat', 'lon', 'n_metal_types',
                                                      'total_metal_genes'])], axis=1)

# Normalise genus name
mags['genus_lower'] = mags['genus'].str.lower().str.strip()

# Merge NGSA and CMMI for Cu/Zn/Pb (CMMI extends coverage beyond Australia)
mags['combined_Cu_ppm'] = mags['ngsa_Cu_ppm'].fillna(mags['cmmi_cu_ppm'])
mags['combined_Zn_ppm'] = mags['ngsa_Zn_ppm'].fillna(mags['cmmi_zn_ppm'])
mags['combined_Pb_ppm'] = mags['ngsa_Pb_ppm'].fillna(mags['cmmi_pb_ppm'])

# Metals to compute niche width for
METALS_MAG = {
    'Cu': 'combined_Cu_ppm',
    'Zn': 'combined_Zn_ppm',
    'Pb': 'combined_Pb_ppm',
    'Ni': 'ngsa_Ni_ppm',
    'Co': 'ngsa_Co_ppm',
    'pH': 'ngsa_field_pH',
}

# Per-genus aggregation: mean, SD, n per metal
MIN_MAGS = 5
results = []
for metal, col in METALS_MAG.items():
    grp = mags.dropna(subset=[col]).groupby('genus_lower')[col]
    r = pd.DataFrame({
        f'{metal}_mean': grp.mean(),
        f'{metal}_sd':   grp.std(ddof=1),
        f'{metal}_n':    grp.count(),
    })
    results.append(r)

genus_geo = pd.concat(results, axis=1)

# Apply minimum MAG count filter per metal (set SD to NaN if n < MIN_MAGS)
for metal in METALS_MAG:
    mask = genus_geo[f'{metal}_n'] < MIN_MAGS
    genus_geo.loc[mask, f'{metal}_sd'] = np.nan
    genus_geo.loc[mask, f'{metal}_mean'] = np.nan

genus_geo = genus_geo.reset_index()
genus_geo.to_csv('data/mgnify_genus_geo_niche.csv', index=False)

for metal in METALS_MAG:
    n = genus_geo[f'{metal}_sd'].notna().sum()
    print(f'{metal}: {n} genera with ≥{MIN_MAGS} MAGs with valid data')

Cu: 85 genera with ≥5 MAGs with valid data
Zn: 85 genera with ≥5 MAGs with valid data
Pb: 85 genera with ≥5 MAGs with valid data
Ni: 2 genera with ≥5 MAGs with valid data
Co: 2 genera with ≥5 MAGs with valid data
pH: 0 genera with ≥5 MAGs with valid data


## Block 2 — PGLS: geochemical niche width ~ metal gene density (MGnify MAGs)

In [3]:
traits = pd.read_csv('data/genus_trait_table.csv')
gsize  = pd.read_csv('data/genus_genome_size_gtdb.csv')
pred   = traits.merge(gsize, on='genus_lower')
pred['genome_mb'] = pred['mean_genome_size_bp'] / 1e6
pred['total_per_mb'] = pred['mean_n_metal_types']             / pred['genome_mb']
pred['tier1_per_mb'] = pred['mean_n_defense_clusters']        / pred['genome_mb']
pred['tier2_per_mb'] = pred['mean_n_homeostasis_clusters']    / pred['genome_mb']

# Z-score predictors
for col, zname in [('total_per_mb','ko_per_mb_total_z'),
                   ('tier1_per_mb','ko_per_mb_tier1_z'),
                   ('tier2_per_mb','ko_per_mb_tier2_z')]:
    pred[zname] = (pred[col] - pred[col].mean()) / pred[col].std()

# Merge with geo niche
geo_niche = pd.read_csv('data/mgnify_genus_geo_niche.csv')
combined  = pred.merge(geo_niche, on='genus_lower')

def run_pgls(df_input, response_col, outfile, tree=TREE, pgls=PGLS_R, rscript=RSCRIPT):
    """Run PGLS for one response variable against all ko_per_mb predictors."""
    df = df_input[['genus_lower', response_col,
                   'ko_per_mb_total_z', 'ko_per_mb_tier1_z', 'ko_per_mb_tier2_z']].copy()
    df = df.dropna()
    df = df.rename(columns={response_col: 'biome_H_std'})
    with tempfile.TemporaryDirectory() as tmp:
        inp = os.path.join(tmp, 'pgls_input.csv')
        df.to_csv(inp, index=False)
        r = subprocess.run([rscript, pgls, inp, tree, outfile],
                          capture_output=True, text=True, cwd=BASE)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-2000:])
    return pd.read_csv(outfile)

# Run PGLS for each metal's SD
mag_pgls_results = []
for metal in METALS_MAG:
    sd_col = f'{metal}_sd'
    # Z-score the SD (log-transform first for skewed distributions)
    vals = combined[sd_col].dropna()
    if len(vals) < 30:
        print(f'{metal}: too few genera ({len(vals)}), skipping')
        continue
    log_sd = np.log1p(combined[sd_col])  # log(1 + SD) stabilises variance
    combined[f'{metal}_sd_z'] = (log_sd - log_sd.mean()) / log_sd.std()
    
    outfile = f'data/mgnify_geo_{metal.lower()}_sd_pgls.csv'
    try:
        res = run_pgls(combined, f'{metal}_sd_z', outfile)
        res['metal'] = metal
        res['dataset'] = 'MGnify MAG'
        res['niche_axis'] = 'geochemical_width_SD'
        mag_pgls_results.append(res)
        print(f'{metal}: n={res["n_taxa"].iloc[0]}, β={res["beta"].iloc[0]:.4f}, p={res["p_value"].iloc[0]:.4f}')
    except Exception as e:
        print(f'{metal}: PGLS failed — {e}')

if mag_pgls_results:
    mag_pgls_df = pd.concat(mag_pgls_results, ignore_index=True)
    # FDR correction across metals (total predictor only)
    total_rows = mag_pgls_df[mag_pgls_df['predictor'] == 'ko_per_mb_total_z'].copy()
    m = len(total_rows)
    total_rows_sorted = total_rows.sort_values('p_value')
    total_rows_sorted['rank'] = range(1, m + 1)
    total_rows_sorted['q_BH'] = (total_rows_sorted['p_value'] * m
                                  / total_rows_sorted['rank']).clip(upper=1.0)
    total_rows_sorted['q_BH'] = total_rows_sorted['q_BH'].cummin()[::-1].values
    mag_pgls_df = mag_pgls_df.merge(total_rows_sorted[['metal','q_BH']], on='metal', how='left')
    mag_pgls_df.to_csv('data/mgnify_geo_niche_pgls.csv', index=False)
    print('\nMGnify geo niche PGLS saved.')
else:
    print('No PGLS results generated.')

Cu: too few genera (19), skipping
Zn: too few genera (19), skipping
Pb: too few genera (20), skipping
Ni: too few genera (1), skipping
Co: too few genera (1), skipping
pH: too few genera (0), skipping
No PGLS results generated.


## Block 3 — AusMicrobiome geochemical niche width (all metals)

In [4]:
# Load sample-level NGSA data
ngsa = pd.read_csv('data/aus_microbiome/aus_sample_ngsa.csv')
ngsa['Sample_ID'] = ngsa['Sample_ID'].astype(str).str.split('/').str[-1]

AUS_METALS = ['Cu', 'Zn', 'Pb', 'Ni', 'Co', 'As', 'Cr', 'Hg', 'pH']
AUS_COLS   = {m: f'ngsa_{m}_ppm' if m != 'pH' else 'ngsa_field_pH' for m in AUS_METALS}

# Filter to samples within 200 km of NGSA site
ngsa_near = ngsa[ngsa['ngsa_dist_km'] <= 200].copy()
print(f'Samples within 200 km of NGSA: {len(ngsa_near)}')

# Load OTU table and taxonomy
print('Loading OTU table...')
otu = pd.read_csv('data/aus_microbiome/BASE_16S_OTU.csv.gz', index_col=0)
otu.index = otu.index.astype(str)

# Use pre-converted CSV (openpyxl not installed in bio_env kernel)
tax = pd.read_csv('data/aus_microbiome/BASE_16S_taxonomy.csv')
tax['OTUId'] = tax['OTUId'].astype(str)
tax['genus_clean'] = (tax['genus']
                      .str.replace(r'^g__', '', regex=True)
                      .str.strip()
                      .replace({'unclassified': np.nan, '': np.nan}))
tax['genus_lower'] = tax['genus_clean'].str.lower()
tax = tax.dropna(subset=['genus_lower'])

# Keep OTUs with genus assignment
otu_g = otu[otu.index.isin(tax['OTUId'])]
print(f'OTUs with genus: {len(otu_g):,}')

# Sample IDs present in both OTU table and NGSA-matched samples
otu_samples = set(otu.columns)
ngsa_samples = set(ngsa_near['Sample_ID'])
shared = sorted(otu_samples & ngsa_samples)
print(f'Shared samples (OTU ∩ NGSA ≤200km): {len(shared)}')

otu_sub  = otu_g[shared]
ngsa_sub = ngsa_near.set_index('Sample_ID').loc[shared]

# Map OTU → genus (map on otu_sub.index directly)
genus_map = tax.set_index('OTUId')['genus_lower']
otu_sub_named = otu_sub.copy()
otu_sub_named.index = otu_sub.index.map(genus_map)
otu_sub_named = otu_sub_named[~otu_sub_named.index.isna()]

# Presence/absence per genus per sample (genus detected if any OTU > 0)
genus_presence = otu_sub_named.groupby(level=0).sum() > 0
print(f'Genera with any detection: {len(genus_presence)}')

Samples within 200 km of NGSA: 1307
Loading OTU table...


OTUs with genus: 18,152
Shared samples (OTU ∩ NGSA ≤200km): 745
Genera with any detection: 933


In [5]:
# Compute per-genus SD for each metal across detection sites
MIN_SITES = 5
aus_geo_rows = []

sample_cols = list(shared)

for metal in AUS_METALS:
    col = AUS_COLS[metal]
    metal_vals = ngsa_sub[col].reindex(sample_cols).values.astype(float)  # per-sample metal value
    valid_mask = ~np.isnan(metal_vals)
    
    means, sds, ns = [], [], []
    for genus in genus_presence.index:
        pres = genus_presence.loc[genus, sample_cols].values  # bool
        site_vals = metal_vals[pres & valid_mask]
        n = len(site_vals)
        if n >= MIN_SITES:
            means.append(np.mean(site_vals))
            sds.append(np.std(site_vals, ddof=1))
            ns.append(n)
        else:
            means.append(np.nan)
            sds.append(np.nan)
            ns.append(n)
    
    aus_geo_rows.append(pd.DataFrame({
        'genus_lower': genus_presence.index,
        f'{metal}_mean': means,
        f'{metal}_sd': sds,
        f'{metal}_n': ns,
    }))

# Combine all metals
aus_geo = aus_geo_rows[0]
for df in aus_geo_rows[1:]:
    aus_geo = aus_geo.merge(df, on='genus_lower')

aus_geo.to_csv('data/aus_genus_geo_niche.csv', index=False)

for metal in AUS_METALS:
    n = aus_geo[f'{metal}_sd'].notna().sum()
    print(f'{metal}: {n} genera with ≥{MIN_SITES} detection sites')

Cu: 778 genera with ≥5 detection sites
Zn: 772 genera with ≥5 detection sites
Pb: 779 genera with ≥5 detection sites
Ni: 779 genera with ≥5 detection sites
Co: 779 genera with ≥5 detection sites
As: 779 genera with ≥5 detection sites
Cr: 779 genera with ≥5 detection sites
Hg: 770 genera with ≥5 detection sites
pH: 0 genera with ≥5 detection sites


## Block 4 — PGLS: geochemical niche width ~ metal gene density (AusMicrobiome)

In [6]:
aus_geo = pd.read_csv('data/aus_genus_geo_niche.csv')
combined_aus = pred.merge(aus_geo, on='genus_lower')

aus_pgls_results = []
for metal in AUS_METALS:
    sd_col = f'{metal}_sd'
    vals = combined_aus[sd_col].dropna()
    if len(vals) < 30:
        print(f'{metal}: too few genera ({len(vals)}), skipping')
        continue
    log_sd = np.log1p(combined_aus[sd_col])
    combined_aus[f'{metal}_sd_z'] = (log_sd - log_sd.mean()) / log_sd.std()
    
    outfile = f'data/aus_geo_{metal.lower()}_sd_pgls.csv'
    try:
        res = run_pgls(combined_aus, f'{metal}_sd_z', outfile)
        res['metal'] = metal
        res['dataset'] = 'AusMicrobiome'
        res['niche_axis'] = 'geochemical_width_SD'
        aus_pgls_results.append(res)
        print(f'{metal}: n={res["n_taxa"].iloc[0]}, β={res["beta"].iloc[0]:.4f}, p={res["p_value"].iloc[0]:.4f}')
    except Exception as e:
        print(f'{metal}: PGLS failed — {e}')

if aus_pgls_results:
    aus_pgls_df = pd.concat(aus_pgls_results, ignore_index=True)
    # BH FDR across metals
    total_rows = aus_pgls_df[aus_pgls_df['predictor'] == 'ko_per_mb_total_z'].copy()
    m = len(total_rows)
    total_rows_sorted = total_rows.sort_values('p_value')
    total_rows_sorted['rank'] = range(1, m + 1)
    total_rows_sorted['q_BH'] = (total_rows_sorted['p_value'] * m
                                  / total_rows_sorted['rank']).clip(upper=1.0)
    total_rows_sorted['q_BH'] = total_rows_sorted['q_BH'].cummin()[::-1].values
    aus_pgls_df = aus_pgls_df.merge(total_rows_sorted[['metal','q_BH']], on='metal', how='left')
    aus_pgls_df.to_csv('data/aus_geo_niche_pgls.csv', index=False)
    print('\nAusMicrobiome geo niche PGLS saved.')

Cu: n=460, β=0.0829, p=0.0120


Zn: n=458, β=-0.1012, p=0.0555


Pb: PGLS failed — Error in data.frame(response = response_col, predictor = predictor_col,  : 
  arguments imply differing number of rows: 1, 0
Calls: Filter ... unlist -> lapply -> lapply -> FUN -> run_pgls -> data.frame
Execution halted



Ni: n=461, β=-0.1061, p=0.0920


Co: n=461, β=-0.1017, p=0.0920


As: n=461, β=-0.1300, p=0.0344


Cr: n=461, β=-0.1887, p=0.0021


Hg: n=454, β=-0.0784, p=0.2136
pH: too few genera (0), skipping

AusMicrobiome geo niche PGLS saved.


## Block 5 — Hotspot occupancy

In [7]:
from sklearn.neighbors import BallTree

hotspots = pd.read_csv('data/hotspots_5grid_filtered.csv')
# Hotspot centroids (5° bins — centroid = bin + 2.5°)
hs_lat = (hotspots['lat_bin'] + 2.5).values
hs_lon = (hotspots['lon_bin'] + 2.5).values
hs_coords = np.deg2rad(np.column_stack([hs_lat, hs_lon]))

# MAG coordinates
mags_valid = mags.dropna(subset=['lat', 'lon']).copy()
mag_coords = np.deg2rad(mags_valid[['lat', 'lon']].values)

# BallTree nearest-neighbour (haversine)
R_KM = 6371.0
HOTSPOT_RADIUS_KM = 350.0  # ~3 degrees at equator
tree_hs = BallTree(hs_coords, metric='haversine')
dists, _ = tree_hs.query(mag_coords, k=1)
mags_valid['dist_hotspot_km'] = dists[:, 0] * R_KM
mags_valid['in_hotspot'] = mags_valid['dist_hotspot_km'] <= HOTSPOT_RADIUS_KM

print(f'MAGs in hotspot zones: {mags_valid["in_hotspot"].sum():,} / {len(mags_valid):,}')
print(f'Hotspot fraction: {mags_valid["in_hotspot"].mean():.3f}')

# Per-genus hotspot occupancy fraction
MIN_GEO_MAGS = 5
grp = mags_valid.groupby('genus_lower')
hotspot_frac = pd.DataFrame({
    'hotspot_frac':   grp['in_hotspot'].mean(),
    'n_geo_mags':     grp['in_hotspot'].count(),
}).reset_index()
hotspot_frac = hotspot_frac[hotspot_frac['n_geo_mags'] >= MIN_GEO_MAGS]
print(f'Genera with ≥{MIN_GEO_MAGS} geo-annotated MAGs: {len(hotspot_frac)}')

# Z-score hotspot_frac as response
hotspot_frac['hotspot_frac_z'] = (
    (hotspot_frac['hotspot_frac'] - hotspot_frac['hotspot_frac'].mean())
    / hotspot_frac['hotspot_frac'].std()
)

# PGLS: hotspot_frac ~ ko_per_mb
hs_input = pred.merge(hotspot_frac[['genus_lower','hotspot_frac_z']], on='genus_lower')
print(f'After merge with predictors: n={len(hs_input)}')

outfile_hs = 'data/hotspot_occupancy_pgls.csv'
try:
    hs_res = run_pgls(hs_input, 'hotspot_frac_z', outfile_hs)
    hs_res['metal'] = 'hotspot'
    hs_res['dataset'] = 'MGnify MAG'
    print('Hotspot PGLS:')
    print(hs_res[['predictor','n_taxa','beta','SE','p_value']].to_string(index=False))
except Exception as e:
    print(f'Hotspot PGLS failed: {e}')

MAGs in hotspot zones: 3,672 / 22,356
Hotspot fraction: 0.164
Genera with ≥5 geo-annotated MAGs: 954
After merge with predictors: n=233


Hotspot PGLS:
        predictor  n_taxa      beta       SE  p_value
ko_per_mb_total_z     227 -0.253469 0.071669 0.000492
ko_per_mb_tier1_z     227 -0.076869 0.050907 0.132445
ko_per_mb_tier2_z     227 -0.088723 0.052156 0.090307


## Block 6 — Cross-dataset concordance

In [8]:
# For genera present in both datasets, compare metal SD estimates
mag_niche = pd.read_csv('data/mgnify_genus_geo_niche.csv')
aus_niche = pd.read_csv('data/aus_genus_geo_niche.csv')

both = mag_niche.merge(aus_niche, on='genus_lower', suffixes=('_mag','_aus'))
print(f'Genera in both datasets: {len(both)}')

SHARED_METALS = ['Cu', 'Zn', 'Pb', 'Ni', 'Co']
concordance = []
for metal in SHARED_METALS:
    sd_mag = f'{metal}_sd_mag'
    sd_aus = f'{metal}_sd_aus'
    if sd_mag not in both.columns or sd_aus not in both.columns:
        continue
    sub = both[[sd_mag, sd_aus]].dropna()
    if len(sub) < 10:
        concordance.append({'metal': metal, 'n': len(sub), 'rho': np.nan, 'p': np.nan})
        continue
    rho, p = stats.spearmanr(sub[sd_mag], sub[sd_aus])
    concordance.append({'metal': metal, 'n': len(sub), 'rho': rho, 'p': p})
    print(f'{metal}: n={len(sub)}, Spearman ρ={rho:.3f}, p={p:.4f}')

concordance_df = pd.DataFrame(concordance)
print('\nCross-dataset concordance summary:')
print(concordance_df.to_string(index=False))

Genera in both datasets: 118

Cross-dataset concordance summary:
metal  n  rho   p
   Cu  8  NaN NaN
   Zn  8  NaN NaN
   Pb  9  NaN NaN
   Ni  0  NaN NaN
   Co  0  NaN NaN


## Block 7 — Forest plots: geochemical niche width PGLS across metals

In [9]:
# Forest plot: AusMicrobiome geochemical niche width PGLS
try:
    aus_pgls = pd.read_csv('data/aus_geo_niche_pgls.csv')
except FileNotFoundError:
    print('AusMicrobiome PGLS file not found — run Block 4 first.')
    aus_pgls = None

# Note: MGnify PGLS skipped — too few genera with NGSA/CMMI annotation after
# merging with the GTDB trait table (19-20 genera; below PGLS minimum of 30).

if aus_pgls is not None:
    sub = aus_pgls[aus_pgls['predictor'] == 'ko_per_mb_total_z'].copy()
    sub = sub.sort_values('p_value')
    metals = sub['metal'].tolist()
    betas  = sub['beta'].values
    ses    = sub['SE'].values
    ps     = sub['p_value'].values
    colors = [METAL_COLORS.get(m, '#999') for m in metals]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    y = range(len(metals))
    ax.barh(list(y), betas, xerr=1.96*ses, height=0.6,
            color=colors, alpha=0.85, capsize=4)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(list(y))
    ax.set_yticklabels([
        f"{m}  (n={int(sub[sub['metal']==m]['n_taxa'].iloc[0])}, "
        f"p={'<0.001' if ps[i]<0.001 else f'{ps[i]:.3f}'})"
        for i, m in enumerate(metals)
    ], fontsize=9)
    ax.set_xlabel('β (log-SD of metal concentration ~ metal KO density/Mb)')
    ax.set_title('Geochemical niche width vs. metal gene density per Mb\n(AusMicrobiome + NGSA)',
                 fontweight='bold')
    # Mark FDR-significant metals
    sig_q = sub[sub.get('q_BH', sub['p_value']) < 0.10]['metal'].tolist() if 'q_BH' in sub.columns else []
    for i, m in enumerate(metals):
        if ps[i] < 0.05:
            ax.text(betas[i] + 1.96*ses[i] + 0.003, i, '*', va='center', color='black')
    
    plt.tight_layout()
    fig.savefig(f'{FIGDIR}/fig8_geo_niche_width_forest.png', dpi=150, bbox_inches='tight')
    fig.savefig(f'{FIGDIR}/fig8_geo_niche_width_forest.pdf', bbox_inches='tight')
    plt.close()
    print('fig8 saved.')
    print('\nKey results:')
    print(sub[['metal','n_taxa','beta','SE','p_value']].to_string(index=False))

fig8 saved.

Key results:
metal  n_taxa      beta       SE  p_value
   Cr     461 -0.188711 0.060949 0.002080
   Cu     460  0.082877 0.032857 0.011996
   As     461 -0.129951 0.061242 0.034378
   Zn     458 -0.101209 0.052722 0.055525
   Co     461 -0.101688 0.060216 0.091952
   Ni     461 -0.106075 0.062816 0.091963
   Hg     454 -0.078436 0.062972 0.213569


## Block 8 — Cross-dataset concordance scatter

In [10]:
both = pd.read_csv('data/mgnify_genus_geo_niche.csv').merge(
    pd.read_csv('data/aus_genus_geo_niche.csv'), on='genus_lower', suffixes=('_mag','_aus'))

SHARED_METALS = ['Cu', 'Zn', 'Pb', 'Ni', 'Co']
n_shared = len([m for m in SHARED_METALS
                if f'{m}_sd_mag' in both.columns and f'{m}_sd_aus' in both.columns])

fig, axes = plt.subplots(1, n_shared, figsize=(4 * n_shared, 4))
if n_shared == 1:
    axes = [axes]

ax_idx = 0
for metal in SHARED_METALS:
    sd_mag = f'{metal}_sd_mag'
    sd_aus = f'{metal}_sd_aus'
    if sd_mag not in both.columns or sd_aus not in both.columns:
        continue
    sub = both[['genus_lower', sd_mag, sd_aus]].dropna()
    ax = axes[ax_idx]
    ax_idx += 1
    if len(sub) < 5:
        ax.text(0.5, 0.5, f'n={len(sub)}\n(insufficient)', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title(f'{metal} niche width')
        continue
    
    rho, p = stats.spearmanr(sub[sd_mag], sub[sd_aus])
    color = METAL_COLORS.get(metal, '#999')
    ax.scatter(sub[sd_mag], sub[sd_aus], c=color, alpha=0.5, s=20, edgecolors='none')
    ax.set_xlabel(f'{metal} SD (MAG-based, ppm)', fontsize=9)
    ax.set_ylabel(f'{metal} SD (amplicon-based, ppm)', fontsize=9)
    ax.set_title(f'{metal}\nρ={rho:.2f}, p={p:.3f}, n={len(sub)}', fontsize=10)

plt.suptitle('Cross-dataset concordance: geochemical niche width\n(MGnify MAGs vs AusMicrobiome)',
             fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
fig.savefig(f'{FIGDIR}/fig9_geo_niche_concordance.png', dpi=150, bbox_inches='tight')
fig.savefig(f'{FIGDIR}/fig9_geo_niche_concordance.pdf', bbox_inches='tight')
plt.close()
print('fig9 saved.')

fig9 saved.


## Block 9 — Summary results table

In [11]:
import warnings
warnings.filterwarnings('ignore')

rows = []
# Previous results (hard-coded from completed analyses)
rows.append({'Analysis':'Categorical niche breadth', 'Dataset':'MicrobeAtlas soil',
             'Metric':'Levins B_std', 'n_genera':603, 'beta':-0.023, 'p':0.0002, 'q_BH':'-', 'Direction':'Specialist'})
rows.append({'Analysis':'Compositional niche breadth', 'Dataset':'MGnify MAGs',
             'Metric':'biome Shannon H', 'n_genera':576, 'beta':0.077, 'p':1.3e-13, 'q_BH':'-', 'Direction':'Cosmopolitan'})
rows.append({'Analysis':'Geochemical filtering (Cu mean)', 'Dataset':'AusMicrobiome+NGSA',
             'Metric':'NGSA Cu ppm mean', 'n_genera':482, 'beta':-0.010, 'p':0.019, 'q_BH':'0.069', 'Direction':'Specialist'})

# Hotspot occupancy
try:
    hs = pd.read_csv('data/hotspot_occupancy_pgls.csv')
    hs_total = hs[hs['predictor'] == 'ko_per_mb_total_z'].iloc[0]
    rows.append({
        'Analysis': 'Hotspot occupancy fraction',
        'Dataset': 'MGnify MAG global',
        'Metric': 'fraction MAGs in hotspot cells',
        'n_genera': int(hs_total['n_taxa']),
        'beta': round(hs_total['beta'], 4),
        'p': round(hs_total['p_value'], 6),
        'q_BH': '-',
        'Direction': 'Less in hotspot' if hs_total['beta'] < 0 else 'More in hotspot',
    })
except FileNotFoundError:
    pass

# New: geochemical niche width
for fname, label in [
    ('data/aus_geo_niche_pgls.csv', 'AusMicrobiome NGSA'),
]:
    try:
        df = pd.read_csv(fname)
        sub = df[df['predictor'] == 'ko_per_mb_total_z'].copy()
        m = len(sub)
        sub_sorted = sub.sort_values('p_value').copy()
        sub_sorted['rank'] = range(1, m + 1)
        sub_sorted['q_BH'] = (sub_sorted['p_value'] * m / sub_sorted['rank']).clip(upper=1.0)
        sub_sorted['q_BH'] = sub_sorted['q_BH'].cummin()[::-1].values
        for _, r in sub_sorted.iterrows():
            direction = 'Specialist' if r['beta'] < 0 else 'Generalist'
            rows.append({
                'Analysis': f'Geochemical niche width ({r["metal"]} SD)',
                'Dataset': label,
                'Metric': f'{r["metal"]}_sd (log-z)',
                'n_genera': int(r['n_taxa']),
                'beta': round(r['beta'], 4),
                'p': round(r['p_value'], 4),
                'q_BH': round(r['q_BH'], 3),
                'Direction': direction,
            })
    except FileNotFoundError:
        pass

summary = pd.DataFrame(rows)
summary.to_csv('data/geo_niche_summary.csv', index=False)
print(summary[['Analysis','Dataset','n_genera','beta','p','Direction']].to_string(index=False))
print()
print('Significant results (p<0.05):')
sig = summary[summary['p'].astype(float) < 0.05]
print(sig[['Analysis','Dataset','n_genera','beta','p','Direction']].to_string(index=False))

                       Analysis            Dataset  n_genera    beta            p       Direction
      Categorical niche breadth  MicrobeAtlas soil       603 -0.0230 2.000000e-04      Specialist
    Compositional niche breadth        MGnify MAGs       576  0.0770 1.300000e-13    Cosmopolitan
Geochemical filtering (Cu mean) AusMicrobiome+NGSA       482 -0.0100 1.900000e-02      Specialist
     Hotspot occupancy fraction  MGnify MAG global       227 -0.2535 4.920000e-04 Less in hotspot
Geochemical niche width (Cr SD) AusMicrobiome NGSA       461 -0.1887 2.100000e-03      Specialist
Geochemical niche width (Cu SD) AusMicrobiome NGSA       460  0.0829 1.200000e-02      Generalist
Geochemical niche width (As SD) AusMicrobiome NGSA       461 -0.1300 3.440000e-02      Specialist
Geochemical niche width (Zn SD) AusMicrobiome NGSA       458 -0.1012 5.550000e-02      Specialist
Geochemical niche width (Co SD) AusMicrobiome NGSA       461 -0.1017 9.200000e-02      Specialist
Geochemical niche wi